In [0]:
import requests, re

# SEC requires a real contact string. Use your actual email.
HEADERS = {"User-Agent": "Utkarsh Saraogi utkarshsaraogi2000@gmail.com"}
LANDING = "https://www.sec.gov/data-research/sec-markets-data/financial-statement-data-sets"

r = requests.get(LANDING, headers=HEADERS, timeout=30)
print("status:", r.status_code, "| bytes:", len(r.content))

# Discover the real ZIP URLs -- never hardcode these
hrefs = re.findall(r'href="([^"]+\.zip)"', r.text)
urls = sorted({h if h.startswith("http") else "https://www.sec.gov" + h for h in hrefs})

print("quarters found:", len(urls))
for u in urls[:3] + ["..."] + urls[-3:]:
    print(" ", u)

In [0]:
import io, zipfile, requests, pandas as pd

BASE = "https://www.sec.gov/files/dera/data/financial-statement-data-sets"

SUB_COLS = ["adsh","cik","name","form","period","filed","accepted","prevrpt"]
NUM_COLS = ["adsh","tag","version","ddate","qtrs","uom","value","coreg"]

def load_quarter(q):
    z = zipfile.ZipFile(io.BytesIO(
        requests.get(f"{BASE}/{q}.zip", headers=HEADERS, timeout=120).content))
    sub = pd.read_csv(z.open("sub.txt"), sep="\t", usecols=SUB_COLS,
                      dtype={"adsh":str,"cik":"Int64","form":str}, low_memory=False)
    num = pd.read_csv(z.open("num.txt"), sep="\t", usecols=NUM_COLS,
                      dtype={"adsh":str,"tag":str,"version":str,"uom":str,"coreg":str},
                      low_memory=False)
    print(f"{q}: sub={len(sub):,}  num={len(num):,}")
    return sub, num

sub_a, num_a = load_quarter("2022q1")
sub_b, num_b = load_quarter("2024q1")

In [0]:
CORE = ["Revenues","NetIncomeLoss","Assets","Liabilities",
        "StockholdersEquity","OperatingIncomeLoss","CostOfRevenue"]

def resolve(sub, num):
    """Attach cik + acceptance time, filter to comparable facts,
    then keep the latest-accepted value per natural key."""
    df = num.merge(sub[["adsh","cik","form","accepted"]], on="adsh", how="inner")
    df["accepted"] = pd.to_datetime(df["accepted"], errors="coerce")
    df = df[
        (df["uom"] == "USD")
        & (df["coreg"].isna() | (df["coreg"].astype(str).str.strip() == ""))
        & (df["version"].str.startswith("us-gaap", na=False))
        & (df["tag"].isin(CORE))
        & df["value"].notna()
    ]
    # precedence in miniature: latest acceptance wins
    df = (df.sort_values("accepted")
            .drop_duplicates(subset=["cik","tag","ddate","qtrs"], keep="last"))
    return df

a, b = resolve(sub_a, num_a), resolve(sub_b, num_b)
print(f"resolved: 2022q1={len(a):,}  2024q1={len(b):,}")

KEY = ["cik","tag","ddate","qtrs"]
m = a.merge(b, on=KEY, suffixes=("_2022","_2024"))
print(f"keys present in BOTH quarters: {len(m):,}")

m["delta"] = m["value_2024"] - m["value_2022"]
m["pct"] = (m["delta"].abs() / m["value_2022"].abs().replace(0, pd.NA)) * 100

changed = m[m["pct"] > 1.0].copy()
print(f"materially changed (>1%): {len(changed):,}  "
      f"({len(changed)/max(len(m),1)*100:.2f}% of overlap)")

cols = ["name_2024","tag","ddate","qtrs","value_2022","value_2024","pct",
        "adsh_2022","adsh_2024","accepted_2024"]
print("\n--- largest revisions ---")
display(changed.nlargest(15, "pct")[cols])

In [0]:
import numpy as np

# --- fix pct dtype (replace(0, pd.NA) had promoted the column to object) ---
denom = m["value_2022"].abs().to_numpy(dtype="float64")
delta = (m["value_2024"] - m["value_2022"]).abs().to_numpy(dtype="float64")
m["delta"] = m["value_2024"] - m["value_2022"]
m["pct"] = np.where(denom > 0, delta / denom * 100, np.nan).astype("float64")

changed = m[m["pct"] > 1.0].copy()
print(f"overlap={len(m):,}   changed>1%={len(changed):,} "
      f"({len(changed)/max(len(m),1)*100:.2f}%)\n")

# --- 1. is the overlap a biased sample? ---
print("=== overlap by fiscal year of ddate ===")
print(m["ddate"].astype(str).str[:4].value_counts().sort_index().to_string())

# --- 2. does the rate depend on period type? ---
print("\n=== rate by qtrs ===")
print(m.groupby("qtrs")["pct"]
       .agg(n="size", chg_pct=lambda s: (s > 1).mean()*100).round(1).to_string())

# --- 3. does one concept dominate? ---
print("\n=== rate by tag ===")
print(m.groupby("tag")["pct"]
       .agg(n="size", chg_pct=lambda s: (s > 1).mean()*100)
       .round(1).sort_values("n", ascending=False).to_string())

# --- 4. artefact detection ---
ratio = (m["value_2024"] / m["value_2022"].replace(0, np.nan)).abs()
print("\n=== scale artefacts ===")
print(f"  identical (ratio ~1.000) : {((ratio>0.999)&(ratio<1.001)).sum():,}")
for k in [-3, -2, -1, 1, 2, 3]:
    hits = ((ratio > 10**k * 0.99) & (ratio < 10**k * 1.01)).sum()
    if hits:
        print(f"  ratio ~10^{k:<2}           : {hits:,}")
print(f"  sign flips               : "
      f"{(np.sign(m['value_2024']) != np.sign(m['value_2022'])).sum():,}")

# --- 5. magnitude shape: one phenomenon or two? ---
print("\n=== pct distribution ===")
print(pd.cut(m["pct"], [0, 0.01, 1, 5, 25, 100, 1000, 1e12])
        .value_counts().sort_index().to_string())

# --- 6. candidates worth verifying by hand ---
cand = changed[
    (changed["qtrs"] == 4)
    & (changed["pct"].between(5, 50))
    & (changed["value_2022"].abs() > 1e8)
].sort_values("pct", ascending=False)

print(f"\n=== verifiable candidates: {len(cand):,} ===")
display(cand[["name_2024","cik","tag","ddate","value_2022","value_2024","pct"]].head(25))

In [0]:
import io, zipfile, requests, pandas as pd, numpy as np

BASE = "https://www.sec.gov/files/dera/data/financial-statement-data-sets"
_zc = {}
def zf(q):
    if q not in _zc:
        _zc[q] = zipfile.ZipFile(io.BytesIO(
            requests.get(f"{BASE}/{q}.zip", headers=HEADERS, timeout=180).content))
    return _zc[q]

# 1) what does num.txt actually contain?
for q in ["2022q1", "2024q1"]:
    print(q, list(pd.read_csv(zf(q).open("num.txt"), sep="\t", nrows=0).columns))

In [0]:
# 2) reload with segments, carry name, filter to CONSOLIDATED facts only
CORE = ["Revenues","NetIncomeLoss","Assets","Liabilities",
        "StockholdersEquity","OperatingIncomeLoss","CostOfRevenue"]

def blank(s):
    return s.isna() | (s.astype(str).str.strip().isin(["", "nan"]))

def load2(q):
    z = zf(q)
    ncols = list(pd.read_csv(z.open("num.txt"), sep="\t", nrows=0).columns)
    want = [c for c in ["adsh","tag","version","ddate","qtrs","uom",
                        "value","coreg","segments"] if c in ncols]
    num = pd.read_csv(z.open("num.txt"), sep="\t", usecols=want, low_memory=False,
                      dtype={"adsh":str,"tag":str,"version":str,"uom":str,
                             "coreg":str,"segments":str})
    sub = pd.read_csv(z.open("sub.txt"), sep="\t", low_memory=False,
                      usecols=["adsh","cik","name","form","period","accepted","prevrpt"],
                      dtype={"adsh":str,"name":str,"form":str})

    df = num.merge(sub[["adsh","cik","name","form","accepted"]], on="adsh")
    df["accepted"] = pd.to_datetime(df["accepted"], errors="coerce")

    keep = (
        (df["uom"] == "USD")
        & blank(df["coreg"])
        & df["version"].str.startswith("us-gaap", na=False)
        & df["tag"].isin(CORE)
        & df["value"].notna()
    )
    if "segments" in df.columns:
        keep &= blank(df["segments"])          # <-- consolidated totals only
    df = df[keep]

    n0 = len(df)
    df = (df.sort_values("accepted")
            .drop_duplicates(["cik","tag","ddate","qtrs"], keep="last"))
    print(f"{q}: kept={n0:,}  after precedence={len(df):,}")
    return df

a2, b2 = load2("2022q1"), load2("2024q1")

m2 = a2.merge(b2, on=["cik","tag","ddate","qtrs"], suffixes=("_22","_24"))
d = (m2["value_24"] - m2["value_22"]).abs().to_numpy("float64")
dn = m2["value_22"].abs().to_numpy("float64")
with np.errstate(divide="ignore", invalid="ignore"):
    m2["pct"] = np.where(dn > 0, d / dn * 100, np.nan)

print(f"\noverlap={len(m2):,}   changed>1% = "
      f"{(m2['pct']>1).sum():,} ({(m2['pct']>1).mean()*100:.2f}%)")
print(f"sign flips: {(np.sign(m2['value_24'])!=np.sign(m2['value_22'])).sum():,}")
print("\nrate by qtrs:")
print(m2.groupby("qtrs")["pct"].agg(n="size", chg=lambda s:(s>1).mean()*100).round(1).to_string())
print("\nrate by tag:")
print(m2.groupby("tag")["pct"].agg(n="size", chg=lambda s:(s>1).mean()*100).round(1).to_string())
print("\npct distribution:")
print(pd.cut(m2["pct"],[0,0.01,1,5,25,100,1000,1e12]).value_counts().sort_index().to_string())

In [0]:
changed2 = m2[m2["pct"] > 1.0].copy()

cand = changed2[
    (changed2["qtrs"] == 4)
    & (changed2["pct"].between(3, 40))
    & (changed2["value_22"].abs() > 5e8)
].sort_values("value_22", key=abs, ascending=False)

print(f"candidates: {len(cand)}")
display(cand[["name_24","cik","tag","ddate","value_22","value_24","pct"]].head(25))
PICK = cand.index[0]          # change to a company you recognize
r = m2.loc[PICK]

print(r[["name_24","cik","tag","ddate","qtrs","value_22","value_24","pct",
         "adsh_22","adsh_24","accepted_22","accepted_24"]].to_string())
print(f"\n{r['value_22']:,.0f}  ->  {r['value_24']:,.0f}   ({r['pct']:.1f}%)\n")

for lbl, adsh, acc in [("ORIGINAL", r["adsh_22"], r["accepted_22"]),
                       ("REVISED ", r["adsh_24"], r["accepted_24"])]:
    print(f"{lbl}  accepted {acc}")
    print(f"  https://www.sec.gov/Archives/edgar/data/"
          f"{int(r['cik'])}/{str(adsh).replace('-','')}/\n")

In [0]:
buckets = {
    "large_pct_nonrevenue": changed2[(changed2["pct"] > 15)
                                     & (~changed2["tag"].isin(["Revenues"]))
                                     & (changed2["value_22"].abs() > 5e8)],
    "small_material":       changed2[changed2["pct"].between(1, 5)
                                     & (changed2["value_22"].abs() > 1e9)],
    "sign_flip":            changed2[np.sign(changed2["value_24"])
                                     != np.sign(changed2["value_22"])],
    "balance_sheet":        changed2[(changed2["qtrs"] == 0)
                                     & (changed2["pct"] > 10)
                                     & (changed2["value_22"].abs() > 1e9)],
}

for name, df in buckets.items():
    print(f"\n===== {name}  (n={len(df)}) =====")
    display(df.sort_values("value_22", key=abs, ascending=False)
              [["name_24","cik","tag","ddate","qtrs","value_22","value_24","pct"]].head(8))

In [0]:
changed2 = m2[m2["pct"] > 1.0].copy()

cand = changed2[
    (changed2["qtrs"] == 4)
    & (changed2["pct"].between(3, 40))
    & (changed2["value_22"].abs() > 5e8)
].sort_values("value_22", key=abs, ascending=False)

print(f"candidates: {len(cand)}")
display(cand[["name_24","cik","tag","ddate","value_22","value_24","pct"]].head(25))
PICK = cand.index[0]          # change to a company you recognize
r = m2.loc[PICK]

print(r[["name_24","cik","tag","ddate","qtrs","value_22","value_24","pct",
         "adsh_22","adsh_24","accepted_22","accepted_24"]].to_string())
print(f"\n{r['value_22']:,.0f}  ->  {r['value_24']:,.0f}   ({r['pct']:.1f}%)\n")

for lbl, adsh, acc in [("ORIGINAL", r["adsh_22"], r["accepted_22"]),
                       ("REVISED ", r["adsh_24"], r["accepted_24"])]:
    print(f"{lbl}  accepted {acc}")
    print(f"  https://www.sec.gov/Archives/edgar/data/"
          f"{int(r['cik'])}/{str(adsh).replace('-','')}/\n")